<div align="center">

# 🔄 ECS VB&YZ — Drive ↔ GitHub Senkronizasyon

Bu notebook ile:
- **GitHub → Drive:** Tüm materyalleri Drive'ınıza çekin
- **Drive → GitHub:** Düzenlemelerinizi GitHub'a gönderin → Web sayfası otomatik güncellenir

---
*Dr. Murat Altun · ECS Veri Bilimi ve YZ Uzmanlığı Programı*

</div>

## 1. Ayarlar ve Drive Bağlantısı

In [ ]:
# ===== AYARLAR =====
DRIVE_KLASOR = "/content/drive/MyDrive/ECS_VB_YZ_90"  # Drive'daki klasör
GITHUB_REPO  = "DrMuratAltun/VB-YZ-90"                # GitHub repo
BRANCH       = "main"

# Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')
print(f"\nDrive klasörü: {DRIVE_KLASOR}")

---
# BÖLÜM A: GitHub → Drive (İlk Kurulum / Yeni Dosya Çekme)

Aşağıdaki hücreyi çalıştırarak GitHub'daki tüm materyalleri Drive'ınıza kopyalayın.
- İlk seferde tüm dosyalar eklenir
- Sonraki sefer sadece yeni eklenen dosyalar gelir (düzenlemeleriniz korunur)

In [ ]:
import os, shutil

TEMP = "/content/_vb_yz_temp"
if os.path.exists(TEMP):
    shutil.rmtree(TEMP)

!git clone --depth 1 -b {BRANCH} https://github.com/{GITHUB_REPO}.git {TEMP} -q
print("Repo klonlandı.")

os.makedirs(DRIVE_KLASOR, exist_ok=True)

# Notebook'ları kopyala (mevcut dosyaları ATLA)
nb_src = os.path.join(TEMP, "notebooks")
nb_dst = os.path.join(DRIVE_KLASOR, "notebooks")
eklenen, atlanan = 0, 0

for week in sorted(os.listdir(nb_src)):
    ws = os.path.join(nb_src, week)
    if not os.path.isdir(ws): continue
    wd = os.path.join(nb_dst, week)
    os.makedirs(wd, exist_ok=True)
    for f in sorted(os.listdir(ws)):
        if not f.endswith('.ipynb'): continue
        dst = os.path.join(wd, f)
        if os.path.exists(dst):
            atlanan += 1
        else:
            shutil.copy2(os.path.join(ws, f), dst)
            eklenen += 1

# Sunumları kopyala
sunum_src = os.path.join(TEMP, "sunumlar_yeni")
sunum_dst = os.path.join(DRIVE_KLASOR, "sunumlar")
if os.path.exists(sunum_src):
    os.makedirs(sunum_dst, exist_ok=True)
    for f in os.listdir(sunum_src):
        if f.endswith('.pptx'):
            dst = os.path.join(sunum_dst, f)
            if not os.path.exists(dst):
                shutil.copy2(os.path.join(sunum_src, f), dst)

# İzlence
izl = os.path.join(TEMP, "egitim-izlencesi.md")
if os.path.exists(izl):
    shutil.copy2(izl, os.path.join(DRIVE_KLASOR, "egitim-izlencesi.md"))

shutil.rmtree(TEMP)
print(f"\nYeni eklenen: {eklenen} notebook")
print(f"Korunan (mevcut): {atlanan} notebook")
print(f"\n✅ Drive klasörü hazır: {DRIVE_KLASOR}")

---
# BÖLÜM B: Drive → GitHub (Düzenlemelerinizi Yayınlama)

Notebook'ları Drive'da Colab ile düzenledikten sonra aşağıdaki hücreleri çalıştırarak değişiklikleri GitHub'a gönderin.

**Bu ne yapar?**
1. Drive'daki notebook'larınızı GitHub repo'suyla karşılaştırır
2. Değişenleri gösterir
3. Onayınızla GitHub'a push eder
4. Web sayfası (GitHub Pages) otomatik güncellenir

### İlk Seferde: GitHub Token Gerekli
1. [github.com/settings/tokens](https://github.com/settings/tokens) → "Generate new token (classic)"
2. `repo` iznini seçin
3. Token'ı kopyalayın
4. Aşağıdaki hücreye yapıştırın (veya Colab Secrets'a `GITHUB_TOKEN` olarak ekleyin)

In [ ]:
# GitHub token — iki yöntemden birini kullanın:

# Yöntem 1: Colab Secrets (önerilen — güvenli)
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    print("Token Colab Secrets'tan alındı.")
except:
    # Yöntem 2: Doğrudan yapıştırma
    GITHUB_TOKEN = input("GitHub Personal Access Token: ")

# Token'ı doğrula
import urllib.request, json
req = urllib.request.Request("https://api.github.com/user",
    headers={"Authorization": f"token {GITHUB_TOKEN}"})
try:
    resp = urllib.request.urlopen(req)
    user = json.loads(resp.read())["login"]
    print(f"✅ Giriş başarılı: {user}")
except:
    print("❌ Token geçersiz! Lütfen kontrol edin.")

### Değişiklikleri Kontrol Et ve Gönder

In [ ]:
import subprocess, shutil, os

WORK = "/content/_vb_yz_push"
if os.path.exists(WORK):
    shutil.rmtree(WORK)

# Repo'yu klonla (tam, push yapabilmek için)
clone_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
!git clone {clone_url} {WORK} -q 2>/dev/null

# Git ayarları
!cd {WORK} && git config user.name "Dr. Murat Altun" && git config user.email "emurataltun@gmail.com"

# Drive'daki notebook'ları repo'ya kopyala
drive_nb = os.path.join(DRIVE_KLASOR, "notebooks")
repo_nb = os.path.join(WORK, "notebooks")

degisen = []
for week in sorted(os.listdir(drive_nb)):
    wd = os.path.join(drive_nb, week)
    if not os.path.isdir(wd): continue
    for f in sorted(os.listdir(wd)):
        if not f.endswith('.ipynb'): continue
        src = os.path.join(wd, f)
        dst = os.path.join(repo_nb, week, f)
        if os.path.exists(dst):
            # Karşılaştır (boyut farkı varsa değişmiş)
            with open(src, 'rb') as a, open(dst, 'rb') as b:
                if a.read() != b.read():
                    shutil.copy2(src, dst)
                    degisen.append(f"{week}/{f}")
        else:
            # Yeni dosya
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
            degisen.append(f"{week}/{f} (YENİ)")

if not degisen:
    print("Değişiklik yok — Drive ve GitHub zaten senkronize.")
else:
    print(f"{len(degisen)} notebook değişmiş:\n")
    for d in degisen:
        print(f"  📝 {d}")

In [ ]:
# Değişiklikleri GitHub'a gönder
if degisen:
    mesaj = f"Drive'dan güncelleme: {len(degisen)} notebook değişti"
    
    result = subprocess.run(
        f'cd {WORK} && git add notebooks/ && git commit -m "{mesaj}" && git push',
        shell=True, capture_output=True, text=True
    )
    
    if result.returncode == 0:
        print("✅ GitHub'a başarıyla gönderildi!")
        print("\n🌐 Web sayfası birkaç dakika içinde güncellenecek:")
        print("   https://drmurataltun.github.io/VB-YZ-90/")
    else:
        print("❌ Hata oluştu:")
        print(result.stderr)
else:
    print("Gönderilecek değişiklik yok.")

# Temizlik
if os.path.exists(WORK):
    shutil.rmtree(WORK)
    print("\nGeçici dosyalar temizlendi.")

---

## Çalışma Akışı Özeti

```
┌──────────────┐     BÖLÜM A      ┌──────────────┐
│   GitHub     │ ──────────────→   │ Google Drive  │
│  (kaynak)    │   ilk kurulum     │  (çalışma)    │
└──────────────┘                   └──────────────┘
       ↑                                  │
       │          BÖLÜM B                 │
       └──────────────────────────────────┘
          düzenlemelerinizi gönder
                    │
                    ↓
           ┌──────────────┐
           │  Web Sayfası │  ← otomatik güncellenir
           │ GitHub Pages │
           └──────────────┘
```

### Günlük Kullanım:
1. Drive'dan notebook aç → Colab'da düzenle → kaydet (otomatik Drive'a kaydolur)
2. Düzenlemeler bitince bu notebook'u aç → **Bölüm B**'yi çalıştır → GitHub'a gönder
3. Web sayfası 1-2 dakika içinde güncellenir

---

<div align="center">

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

© 2026 Dr. Murat Altun

</div>